# RetailPulse: Customer Intelligence & Purchase Analytics

## Project Overview

This notebook performs end-to-end **Exploratory Data Analysis (EDA)** and **data preparation** on a retail customer shopping behaviour dataset.

The goal is to:
- Understand customer demographics and purchase patterns
- Clean and enrich the raw dataset for downstream analysis
- Engineer meaningful features that capture customer behaviour
- Export the clean dataset to a MySQL database for further querying

## 1. Data Loading

In [15]:
import pandas as pd 

# Load the retail customer dataset into a DataFrame
df = pd.read_csv("customer_shopping_behavior.csv")

## 2. Initial Exploration

In [16]:
# Preview the first 5 rows to understand column names, data types and sample values
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [17]:
# Structural overview: column names, non-null counts and dtypes
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

In [18]:
# Statistical summary for all columns (include='all' covers both numeric and categorical)
# For numeric columns  → count, mean, std, min, quartiles, max
# For categorical columns → count, unique values, top value, frequency of top value
df.describe(include='all')

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
count,3900.000000,3900.000000,3900,3900,3900,3900.000000,3900,3900,3900,3900,3863.000000,3900,3900,3900,3900,3900.000000,3900,3900
unique,NaN,NaN,2,25,4,NaN,50,4,25,4,NaN,2,6,2,2,NaN,6,7
top,NaN,NaN,Male,Blouse,Clothing,NaN,Montana,M,Olive,Spring,NaN,No,Free Shipping,No,No,NaN,PayPal,Every 3 Months
freq,NaN,NaN,2652,171,1737,NaN,96,1755,177,999,NaN,2847,675,2223,2223,NaN,677,584
mean,1950.500000,44.068462,NaN,NaN,NaN,59.764359,NaN,NaN,NaN,NaN,3.750065,NaN,NaN,NaN,NaN,25.351538,NaN,NaN
std,1125.977353,15.207589,NaN,NaN,NaN,23.685392,NaN,NaN,NaN,NaN,0.716983,NaN,NaN,NaN,NaN,14.447125,NaN,NaN
min,1.000000,18.000000,NaN,NaN,NaN,20.000000,NaN,NaN,NaN,NaN,2.500000,NaN,NaN,NaN,NaN,1.000000,NaN,NaN
25%,975.750000,31.000000,NaN,NaN,NaN,39.000000,NaN,NaN,NaN,NaN,3.100000,NaN,NaN,NaN,NaN,13.000000,NaN,NaN
50%,1950.500000,44.000000,NaN,NaN,NaN,60.000000,NaN,NaN,NaN,NaN,3.800000,NaN,NaN,NaN,NaN,25.000000,NaN,NaN
75%,2925.250000,57.000000,NaN,NaN,NaN,81.000000,NaN,NaN,NaN,NaN,4.400000,NaN,NaN,NaN,NaN,38.000000,NaN,NaN


In [19]:
# Count of missing (NaN) values per column
df.isnull().sum()

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

## 3. Missing Value Treatment

#### `Review Rating` column contains missing values. Instead of filling with a global statistic, we use the 'median rating within each product category'.

**Why category wise median?**
- Review ratings vary meaningfully across product categories (e.g., electronics are rated differently from clothing).
- Using a global mean or median would blur these differences and introduce bias.
- **Median** is preferred over mean because it is robust to outliers

In [20]:
# Impute missing Review Ratings using the median rating of each product Category

df['Review Rating'] = df.groupby('Category')['Review Rating'].transform('median')
df['Review Rating'] = df['Review Rating'].fillna(df['Review Rating'])

# groupby('Category')  - splits the DataFrame into groups by category
# ['Review Rating']    - selects the target column within each group
# .transform('median') - computes the group median, returning a full-length
#.transform() method in Python is primarily used within the Pandas library to modify DataFrame or Series elements. 
# It applies a specific function or mathematical operation to every element

In [21]:
# Final null check across all columns to confirm no residual missing data
df.isnull().sum()

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

## 4. Column Standardisation

### Rename Columns to snake_case
Raw column names often contain spaces, mixed casing or special characters that make referencing them in code error prone. We convert all column names to **snake_case**, the Python standard.

In [22]:
# Convert all column names to lowercase and replace spaces with underscores

df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ', '_')
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount_(usd)', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='object')

In [24]:
# The column 'purchase_amount_(usd)' contains parentheses, which can cause issues,  rename to a clean identifier
df = df.rename(columns={"purchase_amount_(usd)":"purchase_amount"})

In [25]:
# Final column names after renaming
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='object')

## 5. Feature Engineering
We create two new features:
1. `age` --> Segment customers into lifecycle stages for demographic analysis 
2. `purchase_frequency_days` --> Convert text frequency labels into numeric day intervals for quantitative comparison 

### 5.1 Age Segmentation — `age_group`

We use **quantile-based binning** (`pd.qcut`) to divide customers into four equally sized demographic segments.

**Why `pd.qcut` over `pd.cut`?**
- `pd.cut` creates bins of equal *width* → can leave some bins sparsely populated if age isn't uniformly distributed.
- `pd.qcut` creates bins with equal *frequency* → each segment contains roughly the same number of customers, making group-level comparisons statistically fair.

In [27]:
#Create a column age_group 
labels = ["Young Adult", "Adult", "Middle-aged", "Senior"]
df["age_group"] = pd.qcut(df['age'], q=4, labels = labels)

#q=4: This defines the number of quantiles. A value of 4 tells Pandas to split the data into quartiles (0-25%, 25-50%, 50-75%, and 75-100%).
#labels=labels: This overrides the default numeric interval labels (like (18, 30]) with custom text categories.

In [28]:
#Preview the new column 
df[['age','age_group']].head(10)

,age,age_group
0,55,Middle-aged
1,19,Young Adult
2,50,Middle-aged
3,21,Young Adult
4,45,Middle-aged
5,46,Middle-aged
6,63,Senior
7,27,Young Adult
8,26,Young Adult
9,57,Middle-aged


### 5.2 Purchase Frequency — `purchase_frequency_days`

The `frequency_of_purchases` column stores human-readable labels like `'Weekly'` or `'Quarterly'`. These are categorical and cannot be used directly in numerical calculations (e.g., computing average days between orders or sorting customers by engagement level).

We map each label to its equivalent **number of days**, creating a continuous numeric variable.

In [29]:
#create column purchase_frequency_days --> Number of days between purchases 

frequency_mapping = {

'Fortnightly': 14,
'Weekly': 7,
'Monthly': 30,
'Quarterly': 90,
'Bi-Weekly': 14,
'Annually': 365,
'Every 3 Months': 90 } 

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

In [30]:
#Preview new column 
df[['purchase_frequency_days','frequency_of_purchases']].head(10)

,purchase_frequency_days,frequency_of_purchases
0,14,Fortnightly
1,14,Fortnightly
2,7,Weekly
3,7,Weekly
4,365,Annually
5,7,Weekly
6,90,Quarterly
7,7,Weekly
8,365,Annually
9,90,Quarterly


## 6. Redundancy Check & Column Removal
We audit columns that appear semantically similar before deciding to drop any.

#### Investigation: `discount_applied` vs `promo_code_used`
Both columns seem to capture whether a customer received a price reduction. We verify if they are truly identical across every row.

In [31]:
# Side-by-side preview to visually inspect whether the two columns differ
df[['discount_applied','promo_code_used']].head(10)

,discount_applied,promo_code_used
0,Yes,Yes
1,Yes,Yes
2,Yes,Yes
3,Yes,Yes
4,Yes,Yes
5,Yes,Yes
6,Yes,Yes
7,Yes,Yes
8,Yes,Yes
9,Yes,Yes


In [32]:
# Programmatic check: are the two columns identical for every single row?
# Returns True if there is zero disagreement across the entire dataset
(df['discount_applied'] == df['promo_code_used']).all()

np.True_

**Finding:** Both columns are perfectly identical across all rows — they encode the same information.
 Retaining both would be redundant. We drop `promo_code_used` and keep `discount_applied` as the canonical column name.

In [33]:
# Drop the redundant column; axis=1 specifies we are removing a column (not a row)
df = df.drop('promo_code_used', axis = 1)

In [34]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='object')

## 7. Export to MySQL

With the dataset cleaned and enriched, we export it to a **MySQL database** using SQLAlchemy — a Python SQL toolkit that abstracts database connections and supports multiple engines (MySQL, PostgreSQL, SQLite, etc.).

**Why MySQL?**
- Enables efficient querying of large datasets without loading everything into memory
- Allows integration with BI tools (Tableau, Power BI) and web applications

> **Setup required:** Ensure a MySQL server is running locally and the target database (`customer_behaviour`) has been created before executing the cells below.


In [35]:
# Install required database connector libraries 
!pip install mysql-connector-python sqlalchemy

   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   -----------------

In [41]:
from sqlalchemy import create_engine

# Step 1: Connect to MySQL
# Replace placeholders with your actual details

username = "root"
password = "root"              # Your MySQL password
host = "localhost"             # If running locally
port = "3306"                  # Default MySQL port
database = "customer_behaviour"  # Database name

engine = create_engine(
    f"mysql+mysqlconnector://{username}:{password}@{host}:{port}/{database}"
)

# Step 2: Load DataFrame into MySQL

table_name = "customer"        # Choose any table name

df.to_sql(
    table_name,
    engine,
    if_exists="replace",       # replace existing table
    index=False
)

print(
    f"Data successfully loaded into table '{table_name}' "
    f"in database '{database}'."
)

Data successfully loaded into table 'customer' in database 'customer_behaviour'.
